# AgentFlow Agent Demo

A fast evaluator-friendly walkthrough of the agent workflow: connect the SDK, check health, answer an order question, read a KPI, explore a natural-language query, stream live events, and compare tenant rate limits.

## 1. Setup

An agent needs a typed client before it can reason over live business data. This cell installs the SDK from the repo, points it at the running API, and reads the API key from the environment instead of hard-coding credentials.

In [1]:
from __future__ import annotations

import json
import os
import socket
import subprocess
import sys
import tempfile
import time
import urllib.request
from pathlib import Path
from pprint import pprint

ROOT = Path.cwd().resolve()
SDK_PATH = ROOT / 'sdk'
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-e', str(SDK_PATH), 'sseclient-py'],
    check=True,
    stdout=subprocess.DEVNULL,
    stderr=subprocess.STDOUT,
)
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(SDK_PATH))

from agentflow import AgentFlowClient

BASE_URL = os.environ.get('AGENTFLOW_BASE_URL', 'http://127.0.0.1:8000')
API_KEY = os.environ['AGENTFLOW_API_KEY']
DEMO_ORDER_ID = os.environ.get('AGENTFLOW_DEMO_ORDER_ID', 'ORD-20260404-1001')
DB_PATH = Path(os.environ.get('AGENTFLOW_DUCKDB_PATH', 'agentflow_demo.duckdb'))
client = AgentFlowClient(BASE_URL, api_key=API_KEY, timeout=30.0)

print(f'SDK ready from: {SDK_PATH}')
print(f'API base URL: {BASE_URL}')
print(f'Demo order id: {DEMO_ORDER_ID}')
print(f'DuckDB path: {DB_PATH}')


SDK ready from: D:\DE_project\sdk
API base URL: http://127.0.0.1:45637
Demo order id: ORD-20260410-2810
DuckDB path: C:\Users\uedom\AppData\Local\Temp\agentflow-notebook-demo-v2a4dnf5.duckdb


## 2. Health Check

A production agent should check the pipeline before answering time-sensitive questions. Even in local mode, the response tells us how fresh the latest validated event is and which parts of the platform are degraded.

In [2]:
health = client.health()
summary = {
    'status': health.status,
    'freshness_seconds': health.freshness_seconds,
    'components': [
        {
            'name': component.name,
            'status': component.status,
            'source': component.source,
            'message': component.message,
        }
        for component in health.components
    ],
}
print(json.dumps(summary, indent=2))


{
  "status": "unhealthy",
  "freshness_seconds": null,
  "components": [
    {
      "name": "kafka",
      "status": "unhealthy",
      "source": "live",
      "message": "Check failed: KafkaError{code=_TRANSPORT,val=-195,str=\"Failed to get metadata: Local: Broker transport failure\"}"
    },
    {
      "name": "flink",
      "status": "unhealthy",
      "source": "live",
      "message": "Check failed: [WinError 10061] No connection could be made because the target machine actively refused it"
    },
    {
      "name": "freshness",
      "status": "unhealthy",
      "source": "live",
      "message": "Check failed: 'charmap' codec can't encode characters in position 62-63: character maps to <undefined>"
    },
    {
      "name": "quality_score",
      "status": "unhealthy",
      "source": "live",
      "message": "Check failed: 'charmap' codec can't encode characters in position 60-61: character maps to <undefined>"
    }
  ]
}


## 3. Event-Driven Agent

A support agent usually starts from an entity lookup. This demo uses a real order identifier from the seeded local dataset so the SDK returns a typed object instead of a mocked payload.

In [3]:
order = client.get_order(DEMO_ORDER_ID)
answer = (
    f"Order {order.order_id} is currently {order.status}. "
    f"The latest total is ${order.total_amount:.2f} {order.currency}. "
    f"Customer {order.user_id} placed it at {order.created_at.isoformat()}."
)
print(answer)
print(json.dumps(order.model_dump(mode='json'), indent=2))


Order ORD-20260410-2810 is currently pending. The latest total is $239.97 USD. Customer USR-27447 placed it at 2026-04-10T19:25:35.628323.
{
  "order_id": "ORD-20260410-2810",
  "user_id": "USR-27447",
  "status": "pending",
  "total_amount": 239.97,
  "currency": "USD",
  "created_at": "2026-04-10T19:25:35.628323",
  "is_overdue": false
}


## 4. Metric Query

Agents also need a quick path to KPIs without composing SQL. A typed metric call is the shortest route from a business question to a reliable number.

In [4]:
revenue = client.get_metric('revenue', '24h')
print(f"Today's revenue: ${revenue.value:,.2f} {revenue.unit}")
print(json.dumps(revenue.model_dump(mode='json'), indent=2))


Today's revenue: $6,364.14 USD
{
  "metric_name": "revenue",
  "value": 6364.14,
  "unit": "USD",
  "window": "24h",
  "computed_at": "2026-04-10T16:26:00.763214Z",
  "components": null
}


## 5. Natural-Language Query

Not every question maps to a named metric. This cell shows the fallback NL?SQL path, including the generated SQL, so an evaluator can inspect what the agent actually asked the data layer to do.

In [5]:
nl_result = client.query('top 5 products by revenue this week')
print('Generated SQL:')
print(nl_result.sql)
print('\nRows returned:')
pprint(nl_result.answer)
print('\nMetadata:')
pprint(nl_result.metadata)


Generated SQL:
SELECT SUM(total_amount) as revenue FROM orders_v2 WHERE status != 'cancelled' AND created_at >= NOW() - INTERVAL '1 hour'

Rows returned:
[{'revenue': '6364.14'}]

Metadata:
{'data_freshness_seconds': None, 'execution_time_ms': 2, 'rows_returned': 1}


## 6. Streaming

Polling is enough for demos, but agents often need push-based awareness. This stream shows five recent order-related events over SSE using `sseclient-py`.

In [6]:
from sseclient import SSEClient

request = urllib.request.Request(
    f'{BASE_URL}/v1/stream/events?event_type=order',
    headers={'X-API-Key': API_KEY},
)
stream = urllib.request.urlopen(request, timeout=10)
messages = SSEClient(stream)
seen = []
try:
    for message in messages.events():
        if not message.data or not message.data.strip():
            continue
        payload = json.loads(message.data)
        seen.append(payload)
        print(json.dumps(payload, indent=2))
        print('---')
        if len(seen) == 5:
            break
finally:
    messages.close()

print(f'Captured {len(seen)} streamed events.')


{
  "event_id": "eb395809-c0f7-4997-a44e-2c1b70c6a9c8",
  "topic": "events.validated",
  "processed_at": "2026-04-10T19:25:34.937137",
  "event_type": "order.created",
  "entity_id": null,
  "latency_ms": 23
}
---
{
  "event_id": "26c6910d-352a-40d1-a0d2-ebed75170000",
  "topic": "events.validated",
  "processed_at": "2026-04-10T19:25:35.004803",
  "event_type": "order.created",
  "entity_id": null,
  "latency_ms": 23
}
---
{
  "event_id": "bce2d35b-35ae-4c3c-be2c-10443f786852",
  "topic": "events.validated",
  "processed_at": "2026-04-10T19:25:35.031768",
  "event_type": "order.created",
  "entity_id": null,
  "latency_ms": 21
}
---
{
  "event_id": "85d86ae7-69af-4a8c-9080-6737b001b133",
  "topic": "events.validated",
  "processed_at": "2026-04-10T19:25:35.125326",
  "event_type": "order.created",
  "entity_id": null,
  "latency_ms": 23
}
---
{
  "event_id": "b4d69806-dfd4-4100-988d-6292f29c5604",
  "topic": "events.validated",
  "processed_at": "2026-04-10T19:25:35.156655",
  "event_

## 7. Tenant Comparison

To make per-tenant rate limits observable in one notebook run, the demo spins up a second AgentFlow API instance with the same auth module but intentionally lower RPM limits. The query stays the same; only the API key policy changes.

In [7]:
from agentflow.exceptions import RateLimitError


def free_port() -> int:
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
        sock.bind(('127.0.0.1', 0))
        return sock.getsockname()[1]


def wait_for_openapi(url: str) -> None:
    deadline = time.time() + 20
    while time.time() < deadline:
        try:
            with urllib.request.urlopen(f'{url}/openapi.json', timeout=2.0) as response:
                if response.status == 200:
                    return
        except Exception:
            time.sleep(0.5)
    raise RuntimeError(f'Comparison API did not start: {url}')


compare_dir = Path(tempfile.mkdtemp(prefix='agentflow-rate-demo-'))
compare_db = compare_dir / 'compare.duckdb'
usage_db = compare_dir / 'usage.duckdb'
config_path = compare_dir / 'api_keys.json'
config_path.write_text(
    json.dumps(
        {
            'keys': [
                {
                    'key': 'tenant-low-key',
                    'name': 'Support Agent',
                    'tenant': 'acme',
                    'rate_limit_rpm': 2,
                    'allowed_entity_types': None,
                    'created_at': '2026-04-10',
                },
                {
                    'key': 'tenant-high-key',
                    'name': 'Ops Agent',
                    'tenant': 'acme',
                    'rate_limit_rpm': 4,
                    'allowed_entity_types': None,
                    'created_at': '2026-04-10',
                },
            ]
        },
        indent=2,
    ),
    encoding='utf-8',
)
port = free_port()
compare_url = f'http://127.0.0.1:{port}'
compare_env = os.environ.copy()
compare_env.update(
    {
        'DUCKDB_PATH': str(compare_db),
        'AGENTFLOW_API_KEYS_FILE': str(config_path),
        'AGENTFLOW_USAGE_DB_PATH': str(usage_db),
        'KAFKA_BOOTSTRAP_SERVERS': '127.0.0.1:1',
        'FLINK_JOBMANAGER_URL': 'http://127.0.0.1:1',
    }
)
subprocess.run(
    [sys.executable, '-m', 'src.processing.local_pipeline', '--burst', '40'],
    cwd=ROOT,
    env=compare_env,
    check=True,
    stdout=subprocess.DEVNULL,
    stderr=subprocess.STDOUT,
)
proc = subprocess.Popen(
    [
        sys.executable,
        '-m',
        'uvicorn',
        'src.serving.api.main:app',
        '--host',
        '127.0.0.1',
        '--port',
        str(port),
    ],
    cwd=ROOT,
    env=compare_env,
    stdout=subprocess.DEVNULL,
    stderr=subprocess.STDOUT,
)
try:
    wait_for_openapi(compare_url)
    support_client = AgentFlowClient(compare_url, api_key='tenant-low-key', timeout=10.0)
    ops_client = AgentFlowClient(compare_url, api_key='tenant-high-key', timeout=10.0)

    support_outcomes = []
    for attempt in range(1, 4):
        try:
            support_client.get_metric('revenue', '24h')
            support_outcomes.append(f'support attempt {attempt}: ok')
        except RateLimitError as exc:
            support_outcomes.append(
                f'support attempt {attempt}: rate limited (retry_after={exc.retry_after}s)'
            )

    ops_outcomes = []
    for attempt in range(1, 4):
        try:
            ops_client.get_metric('revenue', '24h')
            ops_outcomes.append(f'ops attempt {attempt}: ok')
        except RateLimitError as exc:
            ops_outcomes.append(
                f'ops attempt {attempt}: rate limited (retry_after={exc.retry_after}s)'
            )

    print('\n'.join(support_outcomes + ops_outcomes))
finally:
    proc.terminate()
    try:
        proc.wait(timeout=5)
    except subprocess.TimeoutExpired:
        proc.kill()
        proc.wait()


support attempt 1: ok
support attempt 2: ok
support attempt 3: rate limited (retry_after=60s)
ops attempt 1: ok
ops attempt 2: ok
ops attempt 3: ok
